# Cityguard — śmieci (YOLO12)

Fine-tuning `yolo12n.pt` na Kaggle `aaronvincent6411/litter-detection-plastic-bottle-and-bag` (butelki i worki). Wynik: `best_litter.pt`.

Dziury trenuj osobno w `ml/train_road.ipynb`, najlepiej na drugim koncie Colab.

Środowisko: **GPU T4**. Przy pierwszym pobraniu wgraj `kaggle.json` (Kaggle → Settings → API → Create New Token).


In [ ]:
!pip -q install -U ultralytics kagglehub


In [ ]:
import torch

!nvidia-smi
if not torch.cuda.is_available():
    raise SystemExit("Włącz GPU: Środowisko wykonawcze → Zmień typ środowiska → T4")
print("GPU:", torch.cuda.get_device_name(0))


## Konfiguracja

`MAX_IMAGES = 0` bierze cały zbiór. Zostaw 0, jeśli mieści się w godzinie; inaczej obetnij próbkę.


In [ ]:
DATASET = "aaronvincent6411/litter-detection-plastic-bottle-and-bag"
MAX_IMAGES = 0  # 0 = cały zbiór
EPOCHS = 25
IMGSZ = 640
BATCH = 16
CONF = 0.35
SAVE_TO_DRIVE = False


In [ ]:

import json
import random
import shutil
from pathlib import Path

import kagglehub
import yaml
from PIL import Image
from ultralytics import YOLO

IMAGE_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def peek(root: Path, limit: int = 25) -> None:
    shown = 0
    for path in root.rglob("*"):
        if not path.is_file():
            continue
        print(" ", path.relative_to(root))
        shown += 1
        if shown >= limit:
            print("  ...")
            return


def download(slug: str) -> Path:
    print("Pobieram", slug)
    root = Path(kagglehub.dataset_download(slug))
    print("Kaggle:", root)
    peek(root)
    return root


def link_split(pairs: list[tuple[Path, str]], dest: Path, names: list[str]) -> Path:
    if dest.exists():
        shutil.rmtree(dest)
    for split in ("train", "val"):
        (dest / "images" / split).mkdir(parents=True)
        (dest / "labels" / split).mkdir(parents=True)
    random.Random(42).shuffle(pairs)
    val_count = max(1, int(len(pairs) * 0.1))
    if len(pairs) < 2:
        raise RuntimeError(f"Za mało obrazów z etykietą: {len(pairs)}")
    for index, (image, label) in enumerate(pairs):
        split = "val" if index < val_count else "train"
        stem = f"{index:05d}_{image.stem}"
        target = dest / "images" / split / f"{stem}{image.suffix.lower()}"
        target.symlink_to(image.resolve())
        (dest / "labels" / split / f"{stem}.txt").write_text(label, encoding="utf-8")
    yaml_path = dest / "cityguard.yaml"
    payload = {
        "train": str((dest / "images" / "train").resolve()),
        "val": str((dest / "images" / "val").resolve()),
        "names": {i: name for i, name in enumerate(names)},
    }
    yaml_path.write_text(yaml.safe_dump(payload, sort_keys=False, allow_unicode=True), encoding="utf-8")
    print(f"{dest.name}: {len(pairs) - val_count} train / {val_count} val, klasy {names}")
    return yaml_path


def backup_epoch(weight_name: str):
    def _hook(trainer):
        if not SAVE_TO_DRIVE:
            return
        best = Path(trainer.save_dir) / "weights" / "best.pt"
        if not best.exists():
            return
        target = Path("/content/drive/MyDrive/cityguard")
        target.mkdir(parents=True, exist_ok=True)
        shutil.copy(best, target / weight_name)
        print(f"Drive: {weight_name} po epoce {trainer.epoch + 1}")

    return _hook


def fit_yolo(data_yaml: Path, run_name: str, weight_name: str) -> Path:
    import torch

    def fit(current_batch: int):
        model = YOLO("yolo12n.pt")
        model.add_callback("on_fit_epoch_end", backup_epoch(weight_name))
        model.train(
            data=str(data_yaml),
            epochs=EPOCHS,
            imgsz=IMGSZ,
            batch=current_batch,
            device=0,
            project="/content/runs",
            name=run_name,
            exist_ok=True,
            patience=8,
            workers=2,
            plots=True,
            save_period=5,
        )
        return model

    try:
        model = fit(BATCH)
    except RuntimeError as exc:
        if "out of memory" not in str(exc).lower():
            raise
        print("CUDA OOM. Powtarzam z batch=8.")
        torch.cuda.empty_cache()
        model = fit(8)

    best = Path("/content/runs") / run_name / "weights" / "best.pt"
    out = Path("/content") / weight_name
    shutil.copy(best, out)
    classes = {"task": run_name, "conf": CONF, "names": {str(key): value for key, value in model.names.items()}}
    Path(f"/content/classes_{run_name}.json").write_text(json.dumps(classes, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Wagi:", out)
    return out


def yolo_text(label_path: Path, image: Path) -> str:
    rows = []
    absolute = False
    for line in label_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        cls = int(float(parts[0]))
        x, y, w, h = (float(value) for value in parts[1:5])
        rows.append((cls, x, y, w, h))
        if max(abs(x), abs(y), abs(w), abs(h)) > 1.5:
            absolute = True
    if not rows:
        return ""
    if absolute:
        with Image.open(image) as handle:
            width, height = handle.size
        rows = [(cls, x / width, y / height, w / width, h / height) for cls, x, y, w, h in rows]
    return "\n".join(f"{cls} {x:.6f} {y:.6f} {w:.6f} {h:.6f}" for cls, x, y, w, h in rows) + "\n"


def prepare_litter(dest: Path) -> Path:
    raw = download(DATASET)
    names_file = next(raw.rglob("obj.names"), None)
    names = []
    if names_file is not None:
        names = [line.strip() for line in names_file.read_text(encoding="utf-8", errors="ignore").splitlines() if line.strip()]
    images = {path.stem: path for path in raw.rglob("*") if path.suffix.lower() in IMAGE_EXT}
    labeled = []
    used = set()
    for label_path in raw.rglob("*.txt"):
        if label_path.name.lower() in {"train.txt", "obj.names", "classes.txt", "obj.data"}:
            continue
        image = images.get(label_path.stem)
        if image is None:
            continue
        text = yolo_text(label_path, image)
        if not text:
            continue
        labeled.append((image, text))
        used.update(int(line.split()[0]) for line in text.splitlines())
    if not labeled:
        raise FileNotFoundError("Litter: nie znalazłem par obraz+txt w formacie YOLO")
    if not names:
        names = [f"class_{index}" for index in range(max(used) + 1)]
    elif max(used) >= len(names):
        names.extend(f"class_{index}" for index in range(len(names), max(used) + 1))
    if MAX_IMAGES and len(labeled) > MAX_IMAGES:
        random.Random(42).shuffle(labeled)
        labeled = labeled[:MAX_IMAGES]
    return link_split(labeled, dest, names)


if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

fit_yolo(prepare_litter(Path("/content/datasets/litter")), "litter", "best_litter.pt")


## Pobranie wag

Rozpakuj ZIP do `ml/weights/`. Inferencja szuka `best_litter.pt`.


In [ ]:
import zipfile
from pathlib import Path
from google.colab import files

bundle = Path("/content/cityguard_best_litter.pt.zip")
names = ["best_litter.pt", "classes_litter.json"]
with zipfile.ZipFile(bundle, "w") as archive:
    found = False
    for name in names:
        path = Path("/content") / name
        if path.exists():
            archive.write(path, name)
            found = True
            print("pakuję", name)
        else:
            print("pomijam", name)
if not found:
    raise SystemExit("Nie ma jeszcze wag. Najpierw odpal trening.")
files.download(str(bundle))


## Podgląd na klatkach z demo

Ustaw `RUN_PREVIEW = True` i uruchom komórkę ręcznie. Próg: `CONF` (0.35).


In [ ]:
RUN_PREVIEW = False  # ustaw True, żeby wgrać klatki i narysować boxy

if not RUN_PREVIEW:
    print("Podgląd wyłączony. Ustaw RUN_PREVIEW = True i odpal tę komórkę ponownie.")
else:
    from google.colab import files
    from pathlib import Path
    from ultralytics import YOLO
    preview = Path("/content/preview_images")
    preview.mkdir(exist_ok=True)
    for name, blob in files.upload().items():
        (preview / name).write_bytes(blob)
    weight = Path("/content/best_litter.pt")
    YOLO(str(weight)).predict(
        source=str(preview),
        conf=CONF,
        save=True,
        project="/content/preview_out",
        name="litter",
        exist_ok=True,
    )
    print("Obrazki z boxami: /content/preview_out/litter")
